In [ ]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv('training_data_ready.csv')
df["gruppen_id"] = df["source_file"].str.replace(
    r"_\d+\.csv$", "", regex=True
)

feature_cols = [c for c in df.columns if c.startswith("A_")]

anomalie_daten = []

for gruppen_name, gruppe in df.groupby("gruppen_id"):
    for col in feature_cols:
        q1 = gruppe[col].quantile(0.25)
        q3 = gruppe[col].quantile(0.75)
        iqr = q3 - q1
        median = gruppe[col].median()

        if iqr > 0:
            untere_grenze = q1 - 1.5 * iqr
            obere_grenze = q3 + 1.5 * iqr

            ausreisser_maske = (gruppe[col] < untere_grenze) | (
                gruppe[col] > obere_grenze
            )
            ausreisser_rows = gruppe[ausreisser_maske]

            for idx, row in ausreisser_rows.iterrows():
                abweichung = row[col] - median
                anomalie_daten.append(
                    {
                        "Original_Zeilen_Index": idx,
                        "Gruppen_ID": gruppen_name,
                        "Dateiname": row["source_file"],
                        "Betroffener_Punkt": col,
                        "Gemessener_Wert": round(row[col], 4),
                        "Gruppen_Median": round(median, 4),
                        "Abweichung_Absolut": round(abweichung, 4),
                        "Abweichung_Richtung": (
                            "ZU HOCH" if abweichung > 0 else "ZU NIEDRIG"
                        ),
                        "Objektkategorie": row["Objektkategorie"],
                        "OAufnahme": row["OAufnahme"],
                        "Schwierigkeit": row["Schwierigkeit"],
                    }
                )

df_report = pd.DataFrame(anomalie_daten)

print(
    f"[INFO] Analyse abgeschlossen. {len(df_report)} punktuelle Ausreißer gefunden."
)

[INFO] Analyse abgeschlossen. 19 punktuelle Ausreißer gefunden.


In [10]:
df_report


,Original_Zeilen_Index,Gruppen_ID,Dateiname,Betroffener_Punkt,Gemessener_Wert,Gruppen_Median,Abweichung_Absolut,Abweichung_Richtung,Objektkategorie,OAufnahme,Schwierigkeit
0,31,3eck_L_L,3eck_L_L_5.csv,A_4_front,-7.8904,-8.1990,0.3086,ZU HOCH,2,2,1
1,150,Durch_L_L_D_O,Durch_L_L_D_O_5.csv,A_1_front,0.0902,0.9063,-0.8161,ZU NIEDRIG,5,3,3
2,146,Durch_L_L_D_O,Durch_L_L_D_O_1.csv,A_6_center,-0.3001,-0.3212,0.0211,ZU HOCH,7,2,3
3,146,Durch_L_L_D_O,Durch_L_L_D_O_1.csv,A_6_back,-0.8516,-0.7754,-0.0762,ZU NIEDRIG,7,2,3
4,157,Durch_L_L_D_U,Durch_L_L_D_U_6.csv,A_5_right,0.3417,0.2485,0.0932,ZU HOCH,5,3,3
5,73,Lumi_L_L,Lumi_L_L_1.csv,A_7_right,0.2611,0.2167,0.0444,ZU HOCH,4,2,2
6,83,Lumi_L_L,Lumi_L_L_11.csv,A_7_right,0.2460,0.2167,0.0293,ZU HOCH,4,2,2
7,99,Lumi_V_L_L,Lumi_V_L_L_2.csv,A_1_center,0.3239,0.4072,-0.0833,ZU NIEDRIG,4,2,2
8,99,Lumi_V_L_L,Lumi_V_L_L_2.csv,A_3_center,0.9643,0.4413,0.5231,ZU HOCH,4,2,2
9,102,Lumi_V_L_L,Lumi_V_L_L_5.csv,A_5_center,0.2179,0.2954,-0.0774,ZU NIEDRIG,4,2,2


In [ ]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv('training_data_ready.csv')
df["gruppen_id"] = df["source_file"].str.replace(
    r"_\d+\.csv$", "", regex=True
)

feature_cols = [c for c in df.columns if c.startswith("A_")]

anomalie_daten = []

for gruppen_name, gruppe in df.groupby("gruppen_id"):
    for col in feature_cols:
        median = gruppe[col].median()

        for idx, row in gruppe.iterrows():
            abweichung = row[col] - median
            anomalie_daten.append(
                {
                        "Original_Zeilen_Index": idx,
                        "Gruppen_ID": gruppen_name,
                        "Dateiname": row["source_file"],
                        "Betroffener_Punkt": col,
                        "Gemessener_Wert": round(row[col], 4),
                        "Gruppen_Median": round(median, 4),
                        "Abweichung_Absolut": round(abweichung, 4),
                        "Abweichung_Richtung": (
                            "ZU HOCH" if abweichung > 0 else "ZU NIEDRIG"
                        ),
                        "Objektkategorie": row["Objektkategorie"],
                        "OAufnahme": row["OAufnahme"],
                        "Schwierigkeit": row["Schwierigkeit"],
                }
            )

df_report_all = pd.DataFrame(anomalie_daten)

print(
    f"[INFO] Analyse abgeschlossen. {len(df_report_all)} punktuelle Ausreißer gefunden."
)

[INFO] Analyse abgeschlossen. 5950 punktuelle Ausreißer gefunden.


In [ ]:
import re
import pandas as pd

df["gruppen_id"] = df["source_file"].str.replace(
    r"_\d+\.csv$", "", regex=True
)

feature_cols = [c for c in df.columns if c.startswith("A_")]

df_medians = df.groupby("gruppen_id")[feature_cols].transform("mean")

df_deviations = (df[feature_cols] - df_medians).abs()

df_medians.columns = [f"{c}_mean" for c in feature_cols]
df_deviations.columns = [f"{c}_abs_dev" for c in feature_cols]

interleaved_cols = []
for c in feature_cols:
    interleaved_cols.append(f"{c}_mean")
    interleaved_cols.append(f"{c}_abs_dev")

metadata_cols = [
    "source_file",
    "gruppen_id",
]
existing_metadata = [c for c in metadata_cols if c in df.columns]

auswertungs_df = pd.concat(
    [df[existing_metadata], df_medians, df_deviations], axis=1
)

final_column_order = existing_metadata + interleaved_cols
auswertungs_df = auswertungs_df[final_column_order]

print(
    f"[INFO] Neues Auswertungs-DF mit Mean erstellt. Spaltenanzahl: {len(auswertungs_df.columns)}"
)

[INFO] Neues Auswertungs-DF mit Mean erstellt. Spaltenanzahl: 72


In [ ]:
deviation_cols = [c for c in auswertungs_df.columns if c.endswith("_abs_dev")]

styled_df = (
    auswertungs_df
    .style.format(
        decimal=",",
        precision=4,
    )
    .background_gradient(
        cmap="Reds",
        subset=deviation_cols,
        axis=0,
    )
)

styled_df

In [ ]:
grenzwert = 0.6

aktive_punkte = []
for c in feature_cols:
    dev_col = f"{c}_abs_dev"
    if (auswertungs_df[dev_col] > grenzwert).any(axis=0):
        aktive_punkte.append(c)

relevante_mess_spalten = []
for c in aktive_punkte:
    relevante_mess_spalten.append(f"{c}_mean")
    relevante_mess_spalten.append(f"{c}_abs_dev")

spalten_gefiltert_df = auswertungs_df[existing_metadata + relevante_mess_spalten]

verbliebene_dev_cols = [
    c for c in spalten_gefiltert_df.columns if c.endswith("_abs_dev")
]
maske_zeilen = (spalten_gefiltert_df[verbliebene_dev_cols] > grenzwert).any(
    axis=1
)

df_ultra_kompakt = spalten_gefiltert_df[maske_zeilen]

print(f"Filter-Ergebnis (Grenzwert > {grenzwert}):")
print(
    f" -> Spalten von {len(auswertungs_df.columns)} auf {len(df_ultra_kompakt.columns)} reduziert."
)
print(
    f" -> Zeilen von {len(auswertungs_df)} auf {len(df_ultra_kompakt)} reduziert."
)

Filter-Ergebnis (Grenzwert > 0.6):
 -> Spalten von 72 auf 8 reduziert.
 -> Zeilen von 170 auf 21 reduziert.


In [ ]:
verbliebene_dev_cols = [
    c for c in df_ultra_kompakt.columns if c.endswith("_abs_dev")
]

styled_kompakt = (
    df_ultra_kompakt
    .style.format(
        decimal=",",
        precision=4,
    )
    .background_gradient(
        cmap="Reds",
        subset=verbliebene_dev_cols,
        axis=0,
    )
)

styled_kompakt

,source_file,gruppen_id,A_1_front_mean,A_1_front_abs_dev,A_1_right_mean,A_1_right_abs_dev,A_6_center_mean,A_6_center_abs_dev
15,4eck_g_L_V_O_7.csv,4eck_g_L_V_O,"0,1676","0,6001","0,4404","0,0538","-0,3078","0,0243"
17,4eck_g_V_V_O_3.csv,4eck_g_V_V_O,"0,4181","0,6968","0,4670","0,5661","-0,3427","0,0090"
21,3eck_L_U_2.csv,3eck_L_U,"0,5688","0,7843","0,3453","0,4407","-0,1661","0,0807"
30,3eck_L_L_4.csv,3eck_L_L,"0,2793","0,6097","0,4061","0,4837","-0,1639","0,1425"
35,Rechteck_L_U_2.csv,Rechteck_L_U,"0,3854","0,6138","0,4685","0,3063","-0,2654","0,0305"
43,Rechteck_L_L_3.csv,Rechteck_L_L,"0,5784","0,8101","0,6051","0,1862","-0,2717","0,0304"
53,Rechteck_H_L_L_3.csv,Rechteck_H_L_L,"0,1198","0,7760","0,4518","0,3199","-0,2487","0,0411"
62,Lumi_L_U_3.csv,Lumi_L_U,"0,4898","0,7705","0,3271","0,4033","-0,3684","0,0308"
67,Lumi_L_U_8.csv,Lumi_L_U,"0,4898","0,6027","0,3271","0,3932","-0,3684","0,0051"
72,Lumi_L_U_13.csv,Lumi_L_U,"0,4898","0,6505","0,3271","0,3862","-0,3684","0,0307"
